## PakMart Traders – Model Training

Trains a **LightGBM Regressor** using MLflow to predict `TotalIncludingTax` (PKR) from transaction features.

#### Import Libraries

In [ ]:
import mlflow

mlexperiment_name = 'pakmart_predict_sale_amount'
mlalgorithm_name  = 'lightgbm'
mlmodel_name      = f'{mlexperiment_name}_{mlalgorithm_name}'

mlflow.set_experiment(mlexperiment_name)

#### Sample clean data (50 %)

In [ ]:
SEED = 42

pakmart_clean_sampled_df = spark.table('pakmart_sale_clean').sample(fraction=0.5, seed=SEED)
print(f'Sampled rows: {pakmart_clean_sampled_df.count():,}')

#### Train / Test split and define features

In [ ]:
train_df, test_df = pakmart_clean_sampled_df.randomSplit([0.75, 0.25], seed=SEED)

train_df.cache()
test_df.cache()

print(f'Train: {train_df.count():,} | Test: {test_df.count():,}')

In [ ]:
# Categorical features: territory, buying group, customer category, day of week, season
categorical_features = ['SalesTerritory', 'BuyingGroup', 'Category', 'DayOfWeekName', 'Season']

# Numerical features: quantity, unit price, tax rate, fiscal month, chiller flag
numerical_features   = ['Quantity', 'UnitPrice', 'TaxRate', 'FiscalMonthNumber', 'IsChillerItem', 'IsWeekend', 'IsFriday']

#### Build ML Pipeline

In [ ]:
from pyspark.ml import Pipeline
from synapse.ml.core.platform import *
from pyspark.ml.feature import OneHotEncoder, VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMRegressor

def create_lgbmr_pipeline(categorical_features, numerical_features, hyperparameters):
    strindx = StringIndexer(
        inputCols=categorical_features,
        outputCols=[f'{f}StrIdx' for f in categorical_features]
    ).setHandleInvalid('keep')

    ohe = OneHotEncoder(
        inputCols=strindx.getOutputCols(),
        outputCols=[f'{f}OHEnc' for f in categorical_features]
    )

    assembler = VectorAssembler(
        inputCols=ohe.getOutputCols() + numerical_features,
        outputCol='features'
    )

    lgbmr = LightGBMRegressor(
        objective=hyperparameters['objective'],
        alpha=hyperparameters['alpha'],
        learningRate=hyperparameters['learning_rate'],
        numLeaves=hyperparameters['num_leaves'],
        labelCol='TotalIncludingTax',
        numIterations=hyperparameters['iterations'],
    )

    return Pipeline(stages=[strindx, ohe, assembler, lgbmr])

In [ ]:
from mlflow.models.signature import ModelSignature
from mlflow.types.utils import _infer_schema

def register_spark_ml_model(mlflow_active_run, model, model_name, signature, metrics, hyperparams):
    mlflow.spark.log_model(model, artifact_path=model_name, signature=signature,
                           registered_model_name=model_name, dfs_tmpdir='Files/mlflow/tmp/')
    mlflow.log_params(hyperparams)
    mlflow.log_metrics(metrics)
    model_uri = f"runs:/{mlflow_active_run.info.run_id}/{model_name}"
    print(f'Model URI: {model_uri}')
    return model_uri

#### Model Training

In [ ]:
lgbmr_hyperparameters = {
    'objective':     'regression',
    'alpha':          0.09,
    'learning_rate':  0.05,
    'num_leaves':     64,
    'iterations':     300
}

In [ ]:
if mlflow.active_run() is None:
    mlflow.start_run()
active_run = mlflow.active_run()
print(f'Run ID: {active_run.info.run_id}')

pipeline = create_lgbmr_pipeline(categorical_features, numerical_features, lgbmr_hyperparameters)
model    = pipeline.fit(train_df)

predictions = model.transform(test_df)
predictions.cache()
print(f'Predictions generated for {predictions.count():,} samples')

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol='TotalIncludingTax', predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(labelCol='TotalIncludingTax', predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(labelCol='TotalIncludingTax', predictionCol='prediction', metricName='r2')

rmse = evaluator_rmse.evaluate(predictions)
mae  = evaluator_mae.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)

print(f'RMSE: {rmse:,.2f} PKR')
print(f'MAE:  {mae:,.2f} PKR')
print(f'R²:   {r2:.4f}')

In [ ]:
from mlflow.models.signature import infer_signature

sample_input  = train_df.limit(5).toPandas()
sample_output = predictions.select('prediction').limit(5).toPandas()
signature     = infer_signature(sample_input, sample_output)

model_uri = register_spark_ml_model(
    active_run, model, mlmodel_name, signature,
    {'rmse': rmse, 'mae': mae, 'r2': r2},
    lgbmr_hyperparameters
)